In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from gt_map import GlasserTianParcellator

# --- CONFIG ---
DATA_ROOT = Path("/mnt/movement/users/jaizor/xtra/data/fmri/ds000030/ds000030_R1.0.4/uncompressed")
PHENO_PATH = Path("/mnt/movement/users/jaizor/xtra/data/fmri/ucla/pheno_ucla.csv")
OUTPUT_DIR = Path("/mnt/movement/users/jaizor/xtra/data/fmri/ucla/output")
OUTPUT_DIR.mkdir(exist_ok=True)

TR = 2.0
TARGET_TR = 2.0
TARGET_DURATION = 300.0
N_TIMEPOINTS = int(TARGET_DURATION / TARGET_TR)  # 150

# Load phenotype
pheno = pd.read_csv(PHENO_PATH)
print(f"🦴 Loaded {len(pheno)} subjects.")

def get_fmri_path(eid):
    return DATA_ROOT / f"sub-{eid}" / "func" / f"sub-{eid}_task-rest_bold.nii.gz"

fmri_paths = [str(get_fmri_path(eid)) for eid in pheno['eid'].astype(str)]

# Parcellate
parcellator = GlasserTianParcellator()
print("🚀 Parcellating...")
timeseries_list, valid_idx = parcellator.process_dataset(
    fmri_paths=fmri_paths,
    tr_values=np.full(len(fmri_paths), TR),
    n_jobs=-1,
    target_tr=TARGET_TR,
    target_duration=TARGET_DURATION
)

# ✅ CRITICAL FIX: Only keep subjects with EXACTLY 414 ROIs, then standardize timepoints
clean_ts_list = []
clean_valid_idx = []

for i, ts in enumerate(timeseries_list):
    # First: check ROI count
    if ts.shape[1] != 414:
        print(f"❌ Skipping subject at original index {valid_idx[i]}: ROI count = {ts.shape[1]} (expected 414)")
        continue

    # Second: ensure exactly 150 timepoints
    if ts.shape[0] > N_TIMEPOINTS:
        ts = ts[:N_TIMEPOINTS]  # truncate
    elif ts.shape[0] < N_TIMEPOINTS:
        # Pad with last timepoint (preserves signal trend)
        pad = np.tile(ts[-1:], (N_TIMEPOINTS - ts.shape[0], 1))
        ts = np.vstack([ts, pad])
    # else: already 150 → keep as-is

    # Final safety check
    if ts.shape == (N_TIMEPOINTS, 414):
        clean_ts_list.append(ts)
        clean_valid_idx.append(valid_idx[i])
    else:
        print(f"⚠️ Unexpected shape after padding/truncation: {ts.shape} for subject {valid_idx[i]}")

# Align phenotype
valid_pheno = pheno.iloc[clean_valid_idx].reset_index(drop=True)

# Save
np.savez_compressed(
    OUTPUT_DIR / "fmri_ucla_gtmap.npz",
    data=np.stack(clean_ts_list, axis=0).astype(np.float32),  # (N, 150, 414)
    subject_ids=valid_pheno['eid'].astype(str).values
)

valid_pheno.to_csv(OUTPUT_DIR / "pheno_ucla_final.csv", index=False)

print(f"🎉 Done! Saved {len(clean_ts_list)} subjects with full 414-ROI coverage.")

INFO:gt_map.core:Processing 268 subjects with n_jobs=-1


🦴 Loaded 268 subjects.
🚀 Parcellating...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 40 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:    4.8s
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    4.9s
[Parallel(n_jobs=-1)]: Done  33 tasks      | elapsed:    5.0s
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:    7.2s
[Parallel(n_jobs=-1)]: Done  65 tasks      | elapsed:    7.4s
[Parallel(n_jobs=-1)]: Done  82 tasks      | elapsed:    9.2s
[Parallel(n_jobs=-1)]: Done 101 tasks      | elapsed:    9.6s
[Parallel(n_jobs=-1)]: Done 120 tasks      | elapsed:   10.3s
[Parallel(n_jobs=-1)]: Done 141 tasks      | elapsed:   11.9s
[Parallel(n_jobs=-1)]: Done 162 tasks      | elapsed:   13.5s
[Parallel(n_jobs=-1)]: Done 185 tasks      | elapsed:   14.2s
[Parallel(n_jobs=-1)]: Done 216 out of 268 | elapsed:   16.3s remaining:    3.9s
[Parallel(n_jobs=-1)]: Done 243 out of 268 | elapsed:   17.7s remaining:    1.8s
[Parallel(n_jobs=-1)]: Done 268 out of 268 | elapsed:   18.5s finished
INFO:gt_

❌ Skipping subject at original index 0: ROI count = 400 (expected 414)
❌ Skipping subject at original index 4: ROI count = 378 (expected 414)
❌ Skipping subject at original index 7: ROI count = 398 (expected 414)
❌ Skipping subject at original index 8: ROI count = 404 (expected 414)
❌ Skipping subject at original index 9: ROI count = 412 (expected 414)
❌ Skipping subject at original index 11: ROI count = 393 (expected 414)
❌ Skipping subject at original index 12: ROI count = 408 (expected 414)
❌ Skipping subject at original index 16: ROI count = 412 (expected 414)
❌ Skipping subject at original index 17: ROI count = 407 (expected 414)
❌ Skipping subject at original index 19: ROI count = 412 (expected 414)
❌ Skipping subject at original index 20: ROI count = 405 (expected 414)
❌ Skipping subject at original index 21: ROI count = 373 (expected 414)
❌ Skipping subject at original index 24: ROI count = 401 (expected 414)
❌ Skipping subject at original index 28: ROI count = 406 (expected 41